In [ ]:
!pip install pandas
!pip install numpy

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
flights = pd.read_csv('raw/flight_data_2024.csv', low_memory=False)
print(flights.shape)
flights.head()


In [ ]:
# Select and rename only the columns needed for the model
df = flights[['op_unique_carrier', 'origin', 'dest', 'crs_dep_time', 'day_of_week', 'month', 'distance', 'dep_delay', 'cancelled']].copy()
df.columns = ['airline', 'origin', 'destination', 'crs_dep_time', 'day_of_week', 'month', 'distance', 'dep_delay', 'cancelled']

# Drop cancelled flights (no departure delay outcome)
df = df[df['cancelled'] == 0.0].drop(columns='cancelled')

# Convert dep_delay to binary 15+ minute delay target
df = df.dropna(subset=['dep_delay'])
df['delayed'] = (df['dep_delay'] >= 15).astype(int)
df = df.drop(columns='dep_delay')

# Convert crs_dep_time (e.g. 1738) to dep_hour (17)
df['crs_dep_time'] = pd.to_numeric(df['crs_dep_time'], errors='coerce')
df = df.dropna(subset=['crs_dep_time'])
df['dep_hour'] = (df['crs_dep_time'].astype(int) // 100).astype(int)
df = df[(df['dep_hour'] >= 0) & (df['dep_hour'] <= 23)]
df = df.drop(columns='crs_dep_time')

# Drop rows with missing key features
df = df.dropna(subset=['airline', 'origin', 'destination', 'distance', 'day_of_week', 'month'])
df['distance'] = df['distance'].astype(int)
df['day_of_week'] = df['day_of_week'].astype(int)
df['month'] = df['month'].astype(int)

print(df.shape)
print(df['delayed'].value_counts())
df.head()


In [ ]:
# Strip whitespace from string columns
for col in ['airline', 'origin', 'destination']:
    df[col] = df[col].str.strip()

print('Unique airlines:', df['airline'].nunique())
print('Unique origins:', df['origin'].nunique())
print('Unique destinations:', df['destination'].nunique())

In [ ]:
df.to_csv('processed/flights_processed.csv', index=False)
print('Saved to processed/flights_processed.csv')

In [ ]:
df['Year'].unique()